# Clasificador de Objetivos de Desarrollo Sostenible (ODS)

Este notebook desarrolla una solución basada en técnicas de Procesamiento de Lenguaje Natural (NLP) y Machine Learning para clasificar textos según los 17 ODS.

## Objetivo
Construir un pipeline de machine learning funcional justificando la elección de la preparación de datos, técnica de reducción de la dimensionalidad, y el algoritmo de clasificación (nivel introductorio).

## 1. Configuración del Entorno y Carga de Datos
Primero, importamos las librerías necesarias para la lectura, el procesamiento de texto y el modelado.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression

import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'matplotlib'

A continuación, cargamos el dataset de OSDG-CD. Este contiene alrededor de 40,000 textos etiquetados.

In [2]:
## 2. Preparación de Textos (Bag of Words & TF-IDF)
Para la representación matricial de los textos utilizaremos el modelo **Bag of Words**, ponderado mediante la técnica **TF-IDF (Term Frequency - Inverse Document Frequency)**.
Se instanciará el objeto `TfidfVectorizer` parametrizado con `ngram_range=(1,2)` para contemplar unigramas y bigramas, y así capturar dependencias secuenciales simples. Adicionalmente, se establecerá el hiperparámetro `max_features` a 5000.

> **Nota Técnica:** La limitación a las 5,000 características más representativas se aplica uniformemente en el vectorizador para controlar la dimensionalidad del corpus. Esta decisión de preprocesamiento es crucial para asegurar la viabilidad computacional de los modelos subsiguientes, particularmente del ensamble de árboles (XGBoost), mitigando el riesgo de exceder la memoria RAM disponible o aumentar sustancialmente el coste de inferencia.

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

## 2. Preparación de los Textos (Bag of Words & TF-IDF)
En lugar de usar modelos masivos como Transformers, nos apegaremos a **Bolsa de Palabras (BOW)** ponderado mediante **TF-IDF**.  
Se configurará un `TfidfVectorizer` incluyendo unigramas y bigramas (`ngram_range=(1,2)`) para captar mejor el contexto, pero **limitaremos el vocabulario máximo (`max_features`) a 5000**. 

> **Justificación Técnica:** Limitar las *features* es vital para no explotar la memoria RAM en el paso posterior de XGBoost, lo cual también asegura que el modelo final exportado .pkl pueda correr ligero y de forma ágil en la aplicación interactiva.

In [ ]:
# Separamos las variables independientes (X) y dependientes (y)
X = df['textos']

# Es fundamental codificar las etiquetas de ODS a números (0 a 16) para que XGBoost funcione.
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['ODS'])

# División en validación (Test) preservando el balance de clases (stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Conjunto de entrenamiento: {X_train.shape[0]} muestras")
print(f"Conjunto de prueba: {X_test.shape[0]} muestras")

## 3. Evaluando el Trade-off: Modelado Lineal vs No Lineal

Para justificar nuestra elección de algoritmo e incorporar la etapa de **reducción de dimensionalidad**, compararemos dos enfoques que reflejan un importante *trade-off* típico del Procesamiento de Lenguaje Natural:

1. **Regresión Logística (Línea Base Lineal sin Reducción):** Como modelo puramente lineal (y uno de los más clásicos para texto), puede manejar matrices esparcidas enormes sin mucha carga de memoria. Esto nos permitiría aprovechar el **vocabulario completo** o casi completo (gran contexto literal).
2. **XGBoost con TruncatedSVD (Modelo No Lineal Reducido):** Un ensamblaje de árboles es excelente capturando relaciones complejas entre palabras, pero cada regla dentro del árbol ocupa memoria. A esto sumamos que el espacio original (5.000 características) es demasiado grande para hacer árboles densos rápidos.

La rúbrica indica aplicar un algoritmo de reducción de la dimensionalidad. Aquí es donde **Truncated SVD** entra en juego.
> **Justificación Reducción Dimensionalidad (Truncated SVD):** El temario menciona *el análisis de componentes principales (PCA) y sus variantes*. Standard PCA centrado no es aplicable nativamente a matrices esparcidas (sparse matrices) devueltas por TF-IDF sin agotar docenas de GB de RAM. `TruncatedSVD` es matemáticamente la variante exacta enseñada para datos dispersos (también conocida como LSA o Análisis Semántico Latente). 

Entrenaremos la Regresión Logística directa para ver el Baseline, y luego entrenaremos XGBoost filtrando el texto previamente por una reducción TruncatedSVD a 300 componentes y evaluaremos los resultados finales.

## 3. Discusión Metodológica: Modelado Lineal vs No Lineal

Con el objetivo de clasificar los ODS e integrar el fundamento de reducción de la dimensionalidad, planteamos la comparación de dos aproximaciones clásicas en el Procesamiento de Lenguaje Natural:

1. **Regresión Logística (Línea Base Lineal):** Como modelo lineal estándar para problemas multiclase, presenta una notoria eficiencia computacional al procesar matrices dispersas de alta dimensionalidad (sparse matrices). Esto nos permite entrenar directamente sobre el vocabulario de 5,000 características extrayendo los coeficientes lineales.
2. **XGBoost posterior a reducción SVD (Abordaje No Lineal):** Los métodos de ensamblaje basados en árboles poseen alta capacidad para modelar relaciones no lineales. Sin embargo, su desempeño y tiempo de entrenamiento degradan exponencialmente frente a espacios vectoriales con miles de dimensiones.

Para resolver el problema de la dimensionalidad exigido antes de la ingesta en XGBoost, implementaremos **Truncated SVD**.
> **Justificación de Reducción de Dimensionalidad:** De acuerdo a los conceptos matemáticos sobre Análisis de Componentes Principales (PCA), sabemos que la versión estándar de PCA requiere centrar los datos, acción computacionalmente prohibitiva cuando se trabaja con matrices dispersas (TF-IDF). Por tanto, `TruncatedSVD` (también conocido como Análisis Semántico Latente o LSA) se emplea como el análogo idóneo para datos esparsos, permitiendo reducir la dimensionalidad sin destruir la estructura original de sparsity.

A continuación, procedemos con el entrenamiento de ambos enfoques, comparando su exactitud.

In [ ]:
# Modelo 2: XGBoost con Reducción de Dimensionalidad (Truncated SVD)
pipeline_xgb = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ('svd', TruncatedSVD(n_components=250, random_state=42)), # Reducimos la dimensionalidad para XGBoost
    ('clf', xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42))
])

# En lugar de entrenar directo, usaremos RandomizedSearchCV para Búsqueda de Hiperparámetros de XGBoost (requerido por rúbrica)
param_grid = {
    'clf__max_depth': [3, 5],
    'clf__n_estimators': [50, 100]
}

print("Iniciando Búsqueda de Hiperparámetros (RandomizedSearchCV) para XGBoost...")
random_search = RandomizedSearchCV(
    pipeline_xgb, param_distributions=param_grid, n_iter=3, cv=3, random_state=42, verbose=2, n_jobs=-1
)

random_search.fit(X_train, y_train)
print(f"Mejores parámetros XGBoost: {random_search.best_params_}")

y_pred_xgb = random_search.predict(X_test)
acc_xgb = accuracy_score(y_test, y_pred_xgb)
print(f"Exactitud (Accuracy) XGBoost (con SVD): {acc_xgb:.4f}")

## 3. Discusión Metodológica: Modelado Lineal vs No Lineal

Con el objetivo de clasificar los ODS e integrar el fundamento de reducción de la dimensionalidad, planteamos la comparación de dos aproximaciones clásicas en el Procesamiento de Lenguaje Natural:

1. **Regresión Logística (Línea Base Lineal):** Como modelo lineal estándar para problemas multiclase, presenta una notoria eficiencia computacional al procesar matrices dispersas de alta dimensionalidad (sparse matrices). Esto nos permite entrenar directamente sobre el vocabulario de 5,000 características extrayendo los coeficientes lineales.
2. **XGBoost posterior a reducción SVD (Abordaje No Lineal):** Los métodos de ensamblaje basados en árboles poseen alta capacidad para modelar relaciones no lineales. Sin embargo, su desempeño y tiempo de entrenamiento degradan exponencialmente frente a espacios vectoriales con miles de dimensiones.

Para resolver el problema de la dimensionalidad exigido antes de la ingesta en XGBoost, implementaremos **Truncated SVD**.
> **Justificación de Reducción de Dimensionalidad:** De acuerdo a los conceptos matemáticos sobre Análisis de Componentes Principales (PCA), sabemos que la versión estándar de PCA requiere centrar los datos, acción computacionalmente prohibitiva cuando se trabaja con matrices dispersas (TF-IDF). Por tanto, `TruncatedSVD` (también conocido como Análisis Semántico Latente o LSA) se emplea como el análogo idóneo para datos esparsos, permitiendo reducir la dimensionalidad sin destruir la estructura original de sparsity.

A continuación, procedemos con el entrenamiento de ambos enfoques, comparando su exactitud.

In [ ]:
print("--- Reporte Clasificación (Regresión Logística Base) ---")
print(classification_report(y_test, y_pred_lr, target_names=[str(i) for i in label_encoder.classes_]))

fig, ax = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=False, cmap='Blues', ax=ax[0])
ax[0].set_title('Matriz Confusión - Regresión Logística')
ax[0].set_xlabel('Predicción')
ax[0].set_ylabel('Real')

sns.heatmap(confusion_matrix(y_test, y_pred_xgb), annot=False, cmap='Greens', ax=ax[1])
ax[1].set_title('Matriz Confusión - XGBoost (SVD)')
ax[1].set_xlabel('Predicción')
ax[1].set_ylabel('Real')

plt.show()

## 5. Predicciones Prácticas Muestrales del Test-Set
Elegiremos explícitamente el modelo que demostró menor sobreajuste visual y es más liviano para probarlo en el mundo real. Aquí mostramos las clasificaciones para al menos cuatro textos que no fueron usados en el aprendizaje.

In [ ]:
indices_muestra = [0, 1, 2, 3]
mejores_predicciones = pipeline_lr.predict(X_test.iloc[indices_muestra])
ods_predichos = label_encoder.inverse_transform(mejores_predicciones)
ods_reales = label_encoder.inverse_transform(y_test[indices_muestra])

for idx, texto, ods_p, ods_r in zip(indices_muestra, X_test.iloc[indices_muestra], ods_predichos, ods_reales):
    print("="*80)
    # Imprimimos solo los primeros 250 caracteres del texto para legibilidad
    print(f"TEXTO: {texto[:250]}...")
    print(f"[ODS REAL]: {ods_r} | [ODS PREDICHO]: {ods_p}")

indices_muestra = [0, 1, 2, 3]
Finalmente exportamos el Pipeline ganador y el `LabelEncoder` usando la librería `joblib`. 
Estos artefactos serán consumidos por nuestro script `app.py` que correrá visualmente la predicción en tiempo real.

In [ ]:
joblib.dump(pipeline_lr, 'modelo_ods.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')
print("Modelos exportados correctamente a disco.")